# CHE face preview on the Colab T4 GPU

GPU-only Blender Cycles previews of CHE's face (front, three-quarter, side; 480 px). No CPU fallback, no video, no body.
Every job writes the pictures, a log and an execution receipt (GPU name, Blender version, timings, file hashes) to
**My Drive/CHE_GPU/jobs/**. Finished views are never rendered again: a job is named by its face code, inputs and settings,
and a view is reused only when its picture matches the receipt's hash and the receipt names the GPU.

**Each session: Runtime → Run all.** That is the only manual step; Colab sessions can only be started by you in the
browser, and they end when Google ends them. The last cell is CHE's GPU worker: it takes jobs from CHE's render queue
(in CHE's Cloudflare Worker, so jobs survive disconnects), renders them here, saves everything to Drive, sends the
pictures to CHE's Creation Room, and stops by itself after 45 idle minutes so the GPU is released. A job cut off by a
disconnect is paused and resumes first in your next session, without rendering finished views again.

**One-time setup** (left bar → key icon, Secrets; switch on *Notebook access* for each):
- `GITHUB_TOKEN`: a fine-grained GitHub token for **`Vondada/chey-app` only**, *Contents: **Read-only***. It only downloads CHE's code, inputs and the queue.
- `CHE_GPU_TOKEN`: CHE's worker token. In CHE: Creation Room → GPU jobs → **Pair Colab** (shown once). It opens only
  CHE's GPU worker routes; pairing again replaces it.
- `RESULTS_TOKEN` (optional): a fine-grained token for **only the private repository `Vondada/che-gpu-results`**,
  *Contents: Read and write*. It can only write to that repository, never to `chey-app`. Without it the results stay in Drive.

Runtime → Change runtime type → **T4 GPU**. Allow Google Drive when asked.
Tokens reach git through environment variables only: never printed, never saved to disk or Drive, never on a command line.


In [ ]:
# 1. GPU
import subprocess
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then run again'

In [ ]:
# 2. Google Drive: pictures, logs and receipts persist here between sessions
from google.colab import drive
drive.mount('/content/drive')
import os, time, pathlib, json
ROOT = '/content/drive/MyDrive/CHE_GPU'
for d in ('jobs', 'cache'): os.makedirs(f'{ROOT}/{d}', exist_ok=True)
marker = pathlib.Path(ROOT, 'persist_check.txt'); before = marker.read_text() if marker.exists() else None
marker.write_text(time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))
print('Drive OK. Previous session wrote:', before or '(first run)')

In [ ]:
# 3. CHE's code and inputs from GitHub. The repository is public, so no secret is needed; an optional read-only
#    GITHUB_TOKEN secret is still used if you saved one (never printed or saved).
import base64
from google.colab import userdata
try: TOKEN = (userdata.get('GITHUB_TOKEN') or '').strip()
except Exception: TOKEN = ''
REPO = 'https://github.com/Vondada/chey-app'
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT='0')
if TOKEN:   # as a header through this process's environment: not in .git/config or any command line
    GIT_ENV.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' + base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode())
def sh(cmd, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).replace(TOKEN, '***') if TOKEN else r.stdout + r.stderr
    if r.returncode: raise RuntimeError(out[-2000:])
    return out
SRC = '/content/chey-app'
for ref in ('main', 'feature/che-colab-gpu-preview'):   # main once PR #517 is merged
    sh(f'rm -rf {SRC} && git clone -q --depth 1 --filter=blob:none --sparse -b {ref} {REPO} {SRC}', GIT_ENV)
    sh(f'cd {SRC} && git sparse-checkout set scripts tools/avatar_face tools/avatar_body tools/avatar_portraits', GIT_ENV)
    if os.path.exists(f'{SRC}/scripts/che_gpu_face_preview.py') and os.path.exists(f'{SRC}/tools/avatar_face/render_device.py'): break
print('code:', ref, sh(f'cd {SRC} && git log -1 --format="%h %s"'))
sh(f'rm -rf /content/che-assets && git clone -q --depth 1 -b che-gpu-assets {REPO} /content/che-assets', GIT_ENV)
print(sh('cd /content/che-assets/project && sha256sum *'))


In [ ]:
# 4. Blender 5.0.1 (the version CHE was built in), downloaded once into Drive and checked against blender.org's SHA-256
import hashlib, urllib.request
VER = '5.0.1'; NAME = f'blender-{VER}-linux-x64'; BASE = 'https://download.blender.org/release/Blender5.0'
cache = f'{ROOT}/cache/{NAME}.tar.xz'
if not os.path.exists(cache):
    urllib.request.urlretrieve(f'{BASE}/{NAME}.tar.xz', cache + '.part'); os.replace(cache + '.part', cache)
h = hashlib.sha256()
with open(cache, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
want = [l.split()[0] for l in urllib.request.urlopen(f'{BASE}/blender-{VER}.sha256').read().decode().splitlines() if l.strip().endswith(f'{NAME}.tar.xz')]
if not want or want[0] != h.hexdigest():
    os.remove(cache); raise RuntimeError('Blender download checksum mismatch: removed it, run this cell again')
if not os.path.exists(f'/content/{NAME}/blender'): sh(f'tar -xf {cache} -C /content')
sh('apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libgl1 libegl1 libxxf86vm1 libxfixes3 > /dev/null')
BL = f'/content/{NAME}/blender'
print(sh(f'{BL} --version').splitlines()[0], '| sha256 OK')

In [ ]:
# 5. MakeHuman / MPFB2 base mesh and targets (CC0), pinned to the commit CHE's fit was made with
if not os.path.exists('/content/mpfb2/src/mpfb/data/3dobjs/base.obj'):
    sh('git clone -q --filter=blob:none https://github.com/makehumancommunity/mpfb2 /content/mpfb2 && cd /content/mpfb2 && git checkout -q b1a2dc98776da9d1eea751b159d42cf61778fe2e')
os.environ['MPFB'] = '/content/mpfb2/src/mpfb/data'
print('MPFB ready')

In [ ]:
# 6. Jobs already finished in Drive: show them and upload any not uploaded yet (nothing is rendered again)
from IPython.display import Image, display
try: RT = (userdata.get('RESULTS_TOKEN') or '').strip()
except Exception: RT = ''
def upload(job):
    if not RT: return 'not uploaded: no RESULTS_TOKEN secret (results stay in Drive)'
    r = subprocess.run(['python3', f'{SRC}/scripts/che_gpu_push_results.py', '--job', str(job)],
                       env=dict(os.environ, RESULTS_TOKEN=RT), capture_output=True, text=True)
    return ((r.stdout + r.stderr).replace(RT, '***').strip().splitlines() or ['(no output)'])[-1]
def show(job):
    rec = json.load(open(job / 'receipt.json'))
    print(f"\n{job.name}: {rec.get('status')} | {rec.get('nvidia_smi')} | {rec.get('blender_version')}")
    for v, info in sorted(rec.get('views', {}).items(), key=lambda kv: int(kv[0])):
        print(' ', info['name'], f"{info['render_seconds']} s", info['device']); display(Image(str(job / info['file']), width=240))
jobs = sorted((d for d in pathlib.Path(ROOT, 'jobs').iterdir() if (d / 'receipt.json').exists()), key=lambda d: d.stat().st_mtime)
for job in jobs:
    if json.load(open(job / 'receipt.json')).get('views'):
        show(job); print('  upload:', upload(job))
print(f'\n{len(jobs)} job folder(s) in Drive')


In [ ]:
# 7. CHE's GPU worker: takes jobs from CHE's queue, renders on this GPU, saves to Drive, sends pictures to CHE.
#    Stops after 45 idle minutes; rerun this cell to start it again. Paused jobs resume first; finished views are reused.
try: CT = (userdata.get('CHE_GPU_TOKEN') or '').strip()
except Exception: CT = ''
CHE_URL = 'https://chey-app.henryjavoni.workers.dev'
env = dict(os.environ, CHE_GPU_TOKEN=CT, RESULTS_TOKEN=RT)
p = subprocess.Popen(['python3', f'{SRC}/scripts/che_gpu_worker.py', '--che-url', CHE_URL, '--blender', BL,
                      '--project', '/content/che-assets/project', '--out', f'{ROOT}/jobs', '--idle-minutes', '45'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in p.stdout:
    for secret in (TOKEN, CT, RT):
        if secret: line = line.replace(secret, '***')
    print(line, end='')
    if line.startswith('RECEIPT '): show(pathlib.Path(line.split(' ', 1)[1].strip()).parent)
print('worker exit', p.wait())
